# Gradio UI for Model Fine-tuning with QLoRa

In [ ]:
# Cell 1: Setup and Imports
import os, math, json, pickle, re, random
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_squared_error
from dotenv import load_dotenv

load_dotenv()

# Constants
MIN_TOKENS, MAX_TOKENS, MIN_CHARS = 150, 160, 300
CEILING_CHARS = MAX_TOKENS * 7

# Available models
AVAILABLE_MODELS = {
    "distilgpt2": "DistilGPT2 (Small)",
    "gpt2": "GPT-2 (Medium)",
    "EleutherAI/gpt-neo-125M": "GPT-Neo 125M",
    "bert-base-uncased": "BERT Base",    
    "microsoft/DialoGPT-small": "DialoGPT Small",
    "google/flan-t5-small": "Flan-T5 Small",
    "facebook/opt-125m": "OPT-125M",
    "meta-llama/Meta-Llama-3.1-8B": "Llama-3.1-8B (Requires HF Token)"
}

In [ ]:
# Cell 2: Item Class and Data Loading
class Item:
    tokenizer = None
    PREFIX = "Price is $"
    QUESTION = "How much does this cost to the nearest dollar?"
    REMOVALS = ['"Batteries Included?": "No"', '"Batteries Included?": "Yes"', 
                '"Batteries Required?": "No"', '"Batteries Required?": "Yes"']

    def __init__(self, data, price):
        self.title = data['title']
        self.price = price
        self.prompt = None
        self.include = False
        self.features = {}
        self.parse(data)

    def parse(self, data):
        contents = '\n'.join(data['description'] + data['features'])
        if data['details']:
            for r in self.REMOVALS:
                data['details'] = data['details'].replace(r, "")
            contents += data['details']
        
        if len(contents) > MIN_CHARS and self.tokenizer:
            # Simple cleaning
            text = f"{self.title}\n{contents[:CEILING_CHARS]}"
            text = re.sub(r'[:\[\]"{}【】\s]+', ' ', text).strip()
            tokens = self.tokenizer.encode(text, add_special_tokens=False)[:MAX_TOKENS]
            text = self.tokenizer.decode(tokens)
            self.prompt = f"{self.QUESTION}\n\n{text}\n\n{self.PREFIX}{round(self.price)}.00"
            self.include = True

    def test_prompt(self):
        if self.prompt:
            return self.prompt.split(self.PREFIX)[0] + self.PREFIX
        return f"{self.QUESTION}\n\n{self.title}\n\n{self.PREFIX}"
    
    def finalize_prompt(self, tokenizer):
        if not self.price: return
        contents = f"{self.title}\n{getattr(self, 'details', '')}"
        tokens = tokenizer.encode(contents, add_special_tokens=False)[:MAX_TOKENS]
        text = tokenizer.decode(tokens)
        self.prompt = f"{self.QUESTION}\n\n{text}\n\n{self.PREFIX}{round(self.price)}.00"
        self.include = True

# Load data
try:
    with open('./train_lite.pkl', 'rb') as f: 
        train = pickle.load(f)
    with open('./test_lite.pkl', 'rb') as f: 
        test = pickle.load(f)
    print(f"Loaded {len(train)} train, {len(test)} test items")
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please download train_lite.pkl and test_lite.pkl first")
    train, test = [], []

# Parse features for baseline
for item in train + test:
    try:
        if hasattr(item, 'details') and item.details:
            item.features = json.loads(item.details)
        else:
            item.features = {}
    except:
        item.features = {}

# Simple feature extraction for baseline
def get_text_length(item):
    return len(item.test_prompt())

def get_features(item):
    return {
        'text_length': get_text_length(item)
    }

In [ ]:
# Cell 3: Tester Class and Baseline Models
class Tester:
    def __init__(self, predictor, data=test, size=50):  # Reduced size for speed
        self.predictor = predictor
        self.data = [d for d in data[:size] if hasattr(d, 'price')]
        self.guesses, self.truths, self.errors, self.sles = [], [], [], []

    def run(self):
        if not self.data:
            fig, ax = plt.subplots(figsize=(10,6))
            ax.text(0.5, 0.5, 'No valid test data', ha='center')
            return fig, {'avg_error': 0, 'rmsle': 0, 'hit_rate': 0}
            
        for item in self.data:
            try:
                g = float(self.predictor(item))
                t = float(item.price)
                e = abs(g - t)
                # Handle zero or negative values for log
                log_g = math.log(max(g, 0.01) + 1)
                log_t = math.log(max(t, 0.01) + 1)
                sle = (log_t - log_g)**2
                
                self.guesses.append(g)
                self.truths.append(t)
                self.errors.append(e)
                self.sles.append(sle)
            except Exception as e:
                print(f"Error predicting: {e}")
                continue
        
        if not self.errors:
            fig, ax = plt.subplots(figsize=(10,6))
            ax.text(0.5, 0.5, 'No predictions made', ha='center')
            return fig, {'avg_error': 0, 'rmsle': 0, 'hit_rate': 0}
        
        avg_err = sum(self.errors)/len(self.errors)
        rmsle = math.sqrt(sum(self.sles)/len(self.sles))
        
        # Calculate hit rate
        hits = 0
        for g, t in zip(self.guesses, self.truths):
            e = abs(g - t)
            if e < 5 or (t > 0 and e/t < 0.1):
                hits += 1
        hit_rate = (hits/len(self.errors))*100 if self.errors else 0
        
        # Create plot
        fig, ax = plt.subplots(figsize=(10,6))
        if self.truths and self.guesses:
            max_val = max(max(self.truths), max(self.guesses))
            ax.plot([0, max_val], [0, max_val], 'b--', alpha=0.5, label='Perfect')
            
            # Color points by error
            colors = []
            for g, t in zip(self.guesses, self.truths):
                e = abs(g - t)
                if e < 5 or (t > 0 and e/t < 0.1):
                    colors.append('green')
                elif e < 15 or (t > 0 and e/t < 0.3):
                    colors.append('orange')
                else:
                    colors.append('red')
            
            ax.scatter(self.truths, self.guesses, c=colors, s=20, alpha=0.6)
            
        ax.set_xlabel('Truth ($)')
        ax.set_ylabel('Guess ($)')
        ax.set_title(f'Avg Error: ${avg_err:.2f} | RMSLE: {rmsle:.2f} | Hits: {hit_rate:.1f}%')
        ax.grid(True, alpha=0.3)
        
        return fig, {'avg_error': avg_err, 'rmsle': rmsle, 'hit_rate': hit_rate}

# Baseline models
def random_pricer(item): 
    return random.uniform(5, 100)

# Calculate average price safely
if train:
    avg_price = sum(float(i.price) for i in train if hasattr(i, 'price'))/len(train)
else:
    avg_price = 50.0

def constant_pricer(item): 
    return avg_price

# Linear regression baseline - FIXED: renamed to avoid conflict
if train:
    # Prepare features
    X_train_list = []
    y_train_list = []
    for i in train:
        if hasattr(i, 'price'):
            X_train_list.append([get_text_length(i)])
            y_train_list.append(float(i.price))
    
    if X_train_list:
        lr_model = LinearRegression()
        lr_model.fit(X_train_list, y_train_list)
    else:
        lr_model = None
else:
    lr_model = None

def linear_pricer(item):
    if lr_model is None:
        return avg_price
    try:
        pred = lr_model.predict([[get_text_length(item)]])[0]
        return max(0, pred)
    except:
        return avg_price

In [ ]:
# Cell 4: Training Functions (Fixed)
def train_model(model_name, learning_rate, epochs, batch_size, use_qlora, lora_r, lora_alpha):
    # Prepare data
    Item.tokenizer = AutoTokenizer.from_pretrained(model_name)
    if Item.tokenizer.pad_token is None:
        Item.tokenizer.pad_token = Item.tokenizer.eos_token
    
    # Parse items to create prompts
    for item in train:
        item.finalize_prompt(Item.tokenizer)
        if not hasattr(item, 'prompt') or not item.prompt:
            if hasattr(item, 'details'):
                # Simple parsing
                text = item.title
                if hasattr(item, 'details') and item.details:
                    text += "\n" + str(item.details)[:200]
                tokens = Item.tokenizer.encode(text, add_special_tokens=False)[:MAX_TOKENS]
                text = Item.tokenizer.decode(tokens)
                item.prompt = f"{Item.QUESTION}\n\n{text}\n\n{Item.PREFIX}{round(item.price)}.00"
    
    train_items = [i for i in train if hasattr(i, 'prompt') and i.prompt][:100]  # Limit for speed
    
    if not train_items:
        raise ValueError("No valid training items found")
    
    # Setup device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Load model
    model_kwargs = {}
    if use_qlora and torch.cuda.is_available():
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4'
        )
        model_kwargs['quantization_config'] = bnb_config
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        **model_kwargs,
        torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
        low_cpu_mem_usage=True
    )
    
    # Move model to device
    model = model.to(device)
    
    # Apply LoRA if using QLoRa
    if use_qlora and torch.cuda.is_available():
        model = prepare_model_for_kbit_training(model)
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=lora_r,
            lora_alpha=lora_alpha,
            target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj']
        )
        model = get_peft_model(model, lora_config)
    
    # Prepare dataset
    texts = [i.prompt for i in train_items if i.prompt]
    dataset = Dataset.from_dict({'text': texts})
    
    def tokenize(examples):
        return Item.tokenizer(
            examples['text'], 
            truncation=True, 
            padding='max_length', 
            max_length=128,
            return_tensors=None
        )
    
    tokenized = dataset.map(tokenize, batched=True, remove_columns=['text'])
    
    # Train
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir='./results',
            learning_rate=learning_rate,
            per_device_train_batch_size=batch_size,
            num_train_epochs=epochs,
            logging_steps=10,
            report_to='none',
            remove_unused_columns=False,
            no_cuda=not torch.cuda.is_available(),
            save_strategy='no'
        ),
        train_dataset=tokenized,
        data_collator=DataCollatorForLanguageModeling(Item.tokenizer, mlm=False)
    )
    
    trainer.train()
    
    # Create predictor
    def predictor(item):
        model.eval()
        prompt = item.test_prompt()
        inputs = Item.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.1,
                do_sample=False,
                pad_token_id=Item.tokenizer.eos_token_id
            )
        
        generated = Item.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract price
        try:
            if 'Price is $' in generated:
                price_part = generated.split('Price is $')[-1]
                # Find first number
                numbers = re.findall(r'\d+\.?\d*', price_part)
                if numbers:
                    return float(numbers[0])
        except:
            pass
        
        # Fallback
        return avg_price
    
    return predictor

In [ ]:
# Cell 5: Gradio Interface
def run_experiment(model_name, learning_rate, epochs, batch_size, 
                   use_qlora, lora_r, lora_alpha, run_baselines):
    
    results = {}
    
    # Check CUDA for QLoRa
    if use_qlora and not torch.cuda.is_available():
        fig = plt.figure(figsize=(10,6))
        plt.text(0.5, 0.5, '⚠️ QLoRa requires a GPU. Please disable QLoRa.', 
                ha='center', va='center', wrap=True)
        plt.axis('off')
        return fig, None, "QLoRa requires GPU - disabled"
    
    # Check HF token for Llama
    if 'llama' in model_name.lower() and not os.getenv('HF_TOKEN'):
        fig = plt.figure(figsize=(10,6))
        plt.text(0.5, 0.5, '❌ HF_TOKEN required for Llama. Set in .env file', 
                ha='center', va='center', wrap=True)
        plt.axis('off')
        return fig, None, "HF_TOKEN missing"
    
    # Run baselines if requested
    if run_baselines:
        random.seed(42)
        _, results['Random'] = Tester(random_pricer).run()
        _, results['Constant'] = Tester(constant_pricer).run()
        _, results['Linear'] = Tester(linear_pricer).run()
    
    # Train and evaluate fine-tuned model
    try:
        # Note: epochs and batch_size here are the numeric values from the sliders
        predictor = train_model(
            model_name, learning_rate, int(epochs), int(batch_size),
            use_qlora, lora_r, lora_alpha
        )
        
        fig, metrics = Tester(predictor).run()
        results['Fine-tuned'] = metrics
        
    except Exception as e:
        import traceback
        traceback.print_exc()
        fig = plt.figure(figsize=(10,6))
        plt.text(0.5, 0.5, f'Error: {str(e)[:200]}', ha='center', va='center', wrap=True)
        plt.axis('off')
        return fig, None, f"Training failed: {str(e)[:100]}"
    
    # Create comparison plot
    fig2 = None
    if len(results) > 1:
        fig2, axes = plt.subplots(1, 3, figsize=(15,4))
        models = list(results.keys())
        
        # Average Error
        axes[0].bar(range(len(models)), [results[m]['avg_error'] for m in models])
        axes[0].set_title('Avg Error ($)')
        axes[0].set_xticks(range(len(models)))
        axes[0].set_xticklabels(models, rotation=45, ha='right')
        
        # RMSLE
        axes[1].bar(range(len(models)), [results[m]['rmsle'] for m in models])
        axes[1].set_title('RMSLE')
        axes[1].set_xticks(range(len(models)))
        axes[1].set_xticklabels(models, rotation=45, ha='right')
        
        # Hit Rate
        axes[2].bar(range(len(models)), [results[m]['hit_rate'] for m in models])
        axes[2].set_title('Hit Rate (%)')
        axes[2].set_xticks(range(len(models)))
        axes[2].set_xticklabels(models, rotation=45, ha='right')
        
        plt.tight_layout()
    
    return fig, fig2, f"Training completed! (Epochs: {epochs}, Batch: {batch_size})"

# Create Gradio interface
with gr.Blocks(title='Model Fine-tuning UI', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🚀 Model Fine-tuning for Price Prediction')
    
    with gr.Row():
        with gr.Column(scale=1):
            model_dropdown = gr.Dropdown(
                choices=list(AVAILABLE_MODELS.keys()),
                value=next(iter(AVAILABLE_MODELS)),
                label='Select Model',
                info='First 4 are small, Llama requires HF token'
            )
            
            with gr.Accordion('Training Parameters', open=True):
                lr_input = gr.Number(value=2e-4, label='Learning Rate')
                epochs_slider = gr.Slider(1, 10, value=1, step=1, label='Epochs')
                batch_slider = gr.Slider(1, 8, value=1, step=1, label='Batch Size')
            
            with gr.Accordion('QLoRa Settings (GPU only)', open=False):
                use_qlora_cb = gr.Checkbox(label='Enable QLoRa')
                lora_r_slider = gr.Slider(2, 32, value=8, step=2, label='LoRA Rank')
                lora_alpha_slider = gr.Slider(8, 64, value=16, step=8, label='LoRA Alpha')
            
            run_baselines_cb = gr.Checkbox(label='Compare with Baselines', value=True)
            train_btn = gr.Button('🚀 Start Training', variant='primary')
            status_box = gr.Textbox(label='Status', interactive=False)
        
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem('Model Performance'):
                    plot1 = gr.Plot(label='Predictions vs Truth')
                with gr.TabItem('Comparison'):
                    plot2 = gr.Plot(label='Model Comparison')
    
    train_btn.click(
        run_experiment,
        inputs=[
            model_dropdown, lr_input, epochs_slider, batch_slider, 
            use_qlora_cb, lora_r_slider, lora_alpha_slider, run_baselines_cb
        ],
        outputs=[plot1, plot2, status_box]
    )

In [ ]:
# Cell 6: Launch
if __name__ == '__main__':
    # Check environment
    if not os.getenv('HF_TOKEN'):
        print("⚠️  HF_TOKEN not set. Create .env file with: HF_TOKEN=your_token")
    
    if torch.cuda.is_available():
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠️  No GPU - training will be slow")
    
    # Launch
    demo.launch(
        inbrowser=True,
    )